# OCR Quality Analysis — Post-Extraction

> **Sanitized release.** This notebook is a faithful copy of the original working
> notebook from a federal document-classification project, edited for public
> sharing: cell outputs are stripped, file paths and system names are
> generalized, and the ten real business categories are renamed `CAT-A` … `CAT-J`.
> Cells that were duplicated once per category in the original are collapsed to
> one or two representative copies, marked with a note. Nothing else about the
> structure or approach has been changed.


## Context

The corpus (250,000+ scanned records accumulated over decades — everything
from old typewritten pages digitized long ago to modern computer-generated
PDFs) was OCR'd locally with Tesseract, first pages of each document, across
multiple parallel worker instances. That pass wrote a word-level confidence
table (one CSV per document) as it went.

*The notebook that ran the corpus OCR pass and the DPI benchmarking ("part 1")
was not recovered for this release; its outputs are what this notebook
consumes.* This notebook aggregates those word-level confidences into a
document-level OCR quality score and looks at how quality is distributed
across the corpus — the signal that later rode along with every model
prediction in the review queue.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm

### Compile document-level confidence scores

Tesseract reports a confidence value per recognized word (`-1` for non-word boxes). The document score is the mean of the real word confidences over the OCR'd pages.

In [ ]:
#ran only once, after pt 1 was finished
def extract_scores():
    '''
    Extracts the word-level confidence scores from the per-document files and
    compiles them into a single file.

    :return: A dataframe with the document_id and the conf_avg
    '''
    print('|INFO| begin extraction')
    temp_dir_names = []
    conf_lst = []

    for file in tqdm(glob.glob(r'data/ocr-confidence-scores/*.csv')):
        if 'crash' in file:
            #documents whose OCR pass failed are tracked, not dropped
            conf_lst.append('crash')
        else:
            df = pd.read_csv(file)
            conf = df['conf'].tolist()
            conf = [i for i in conf if i != -1]
            conf_avg = np.mean(conf)
            conf_lst.append(conf_avg)
        file = os.path.basename(file)
        temp_dir_names.append(file)
    #generate file from above loop
    df = pd.DataFrame(list(zip(temp_dir_names, conf_lst)), columns=['document_id', 'conf_avg'])
    df.to_csv('conf_results_frm_indv.csv', index=False)

    return print('|INFO| extraction complete')

extract_scores()

In [ ]:
#ran only once, after pt 1 was finished
#join with the model-prediction master table
df_ref = pd.read_csv('predictions_master_cleaned.csv')
df_conf = pd.read_csv('conf_results_frm_indv_cleaned.csv')
df_merg = pd.merge(df_ref, df_conf, on='document_id')
df_merg.to_csv('conf_results_compiled_with_pred_master.csv', index=False)

In [ ]:
#ran only once, after pt 1 was finished
#join with the review team's metadata-database export
df_left = pd.read_csv('conf_results_compiled_with_pred_master.csv')
df_right = pd.read_csv('metadata_db_export.csv')
df_merg = pd.merge(df_left, df_right, on='directory', how='left')
df_merg.to_csv('conf_results_compiled_with_pred_master_metadata.csv', index=False)

### OCR quality across the corpus

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('conf_results_compiled_with_pred_master.csv')

#remove crashed out files
df = df[df['conf_avg'] != 'crash']
#redefine conf as float
df['conf_avg'] = df['conf_avg'].astype(float)

#all docs
df_all = df['conf_avg'].tolist()

kwargs = dict(histtype='stepfilled', alpha=0.7, density=False, bins=2000)
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 150})
plt.title("All Repository Documents, OCR 'Quality'")
plt.hist(df_all, **kwargs, label='All Docs')

plt.xticks(np.arange(0, 101, 5))
plt.xlabel('OCR Confidence (in %)')
plt.ylabel('Doc Frequency')
plt.legend(loc='upper left', title='Doc Class:')
plt.show()

The distribution had a long tail of degraded / typewritten scans reaching down
below 40% confidence, a broad hump in the mid-to-high 80s, and a sharp spike
around 93–94% where clean, modern computer-generated documents pile up. A
sanitized recreation of this figure is in
[`../figures/ocr-confidence-distribution.svg`](../figures/ocr-confidence-distribution.svg).

### OCR quality by metadata completeness

Subset exploration against the metadata database export — documents marked complete, versus completeness of individual reference fields. (Field names generalized.)

In [ ]:
#ocr quality of documents present in the metadata database
import numpy as np
import matplotlib.pyplot as plt

#load file
df = pd.read_csv('conf_results_compiled_with_pred_master_metadata.csv')

#subset
df_nocrash = df[df['conf_avg'] != 'crash']
df_nocrash['conf_avg'] = df_nocrash['conf_avg'].astype(float)
df_completed = df_nocrash[df_nocrash['record_status'] == 'Completed']
df_no_ref = df_completed[df_completed['reference_id'].isnull()]
df_no_ref_no_code = df_completed[(df_completed['subject_code'].isnull() & (df_completed['reference_id'].isnull()))]
df_no_ref_no_code_no_det = df_completed[(df_completed['subject_code'].isnull() & (df_completed['reference_id'].isnull() & df_completed['subject_detail'].isnull()))]

#prep lists for figures
df_completed_lst = df_completed['conf_avg'].tolist()
df_no_ref_lst = df_no_ref['conf_avg'].tolist()
df_no_ref_no_code_lst = df_no_ref_no_code['conf_avg'].tolist()
df_no_ref_no_code_no_det_lst = df_no_ref_no_code_no_det['conf_avg'].tolist()

#figure params
kwargs = dict(histtype='stepfilled', alpha=0.7, density=False, bins=200)
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 100})
plt.title('OCR "Quality": metadata-database documents')
plt.hist(df_nocrash['conf_avg'].tolist(), **kwargs, label='Total')
plt.hist(df_completed_lst, **kwargs, label='Completed')
plt.hist(df_no_ref_lst, **kwargs, label='Completed & No reference_id')
plt.hist(df_no_ref_no_code_lst, **kwargs, label='Completed & No reference_id & No subject_code')
plt.hist(df_no_ref_no_code_no_det_lst, **kwargs, label='Completed & No reference_id & No subject_code & No subject_detail')

plt.xticks(np.arange(0, 101, 5))
plt.xlabel('OCR Confidence (in %)')
plt.ylabel('Doc Frequency')
plt.legend(loc='upper left', title='Metadata completeness:')
plt.xlim([15, 100])
plt.show()

### OCR quality by predicted category

*Collapsed: the original cell built one block per category. Three are shown; the remaining categories repeated the same pattern.*

In [ ]:
#exploration of OCR quality as a function of ml model prediction results
import matplotlib.pyplot as plt

#load file
df = pd.read_csv('conf_results_compiled_with_pred_master_metadata.csv')

#subset
df_nocrash = df[df['conf_avg'] != 'crash']
df_nocrash['conf_avg'] = df_nocrash['conf_avg'].astype(float)

def conf_column_for(pred_col):
    x = df_nocrash[df_nocrash[pred_col] == 1]
    x = np.array(x['conf_avg'].tolist())
    return x.reshape(len(x), 1)

pred_cat_a = conf_column_for('pred_CAT-A')
pred_cat_b = conf_column_for('pred_CAT-B')
pred_cat_c = conf_column_for('pred_CAT-C')
# ...repeated for the remaining categories in the original notebook

# create the dataframe; enumerate is used to make column names
df_fig = pd.concat(
    [pd.DataFrame(a, columns=[f'x{i}']) for i, a in
     enumerate([pred_cat_a, pred_cat_b, pred_cat_c], 1)], axis=1)

# plot the data
df_fig.plot.hist(stacked=True, bins=100, density=False, figsize=(10, 6), grid=True)
plt.legend(['pred_CAT-A', 'pred_CAT-B', 'pred_CAT-C'], title='Document Type:')
plt.xticks(np.arange(0, 101, 5))
plt.xlabel('OCR Confidence (in %)')
plt.ylabel('Frequency')
plt.title('OCR "Quality": ML Prediction Results')

Categories dominated by older typewritten material sat visibly lower on OCR
confidence than categories that were mostly modern documents — which is why
the review queue carried OCR confidence next to every model score: the same
probability deserves different trust on clean versus garbled text. Sanitized
recreation: [`../figures/ocr-quality-by-category.svg`](../figures/ocr-quality-by-category.svg).

### Confidence-band subset extractor

Utility for pulling documents inside an OCR-confidence band — used to spot-check what, say, 30–50% confidence text actually looked like, and to pick re-OCR candidates.

In [ ]:
#super dooper subset extractor
start_percent = 91
stop_percent = 100

df = pd.read_csv('conf_results_compiled_with_pred_master.csv')

#remove crashed out files
df = df[df['conf_avg'] != 'crash']

#redefine conf as float
df['conf_avg'] = df['conf_avg'].astype(float)

#save
df_out = df.loc[(df['conf_avg'] >= start_percent) & (df['conf_avg'] <= stop_percent),
                ['document_id', 'conf_avg', 'directory']]
df_out = df_out.sort_values(by=['conf_avg'])
df_out.to_csv('df_out_{}_to_{}.csv'.format(start_percent, stop_percent), index=False)